# Drunk Driving Fatality Analysis — Data Preparation
**Author:** Jared Oberg  
**Dataset:** NHTSA Fatality Analysis Reporting System (FARS) via Google BigQuery  
**Years:** 2015–2016  

---

## Project Overview

Drunk driving remains one of the leading causes of traffic fatalities in the United States.
This project analyzes **18,130 alcohol-related crashes** across 2015–2016 using the NHTSA FARS dataset
to identify geographic patterns and key risk factors associated with drunk driving fatalities.

### How This Project Started

The project originally set out to explore the full FARS dataset across 2015–2020 — all fatal traffic
crashes regardless of cause. During initial exploratory analysis, one pattern stood out immediately:
**approximately 27% of all traffic fatalities involved a drunk driver**. Given the scale and consistency
of that finding, the project focus shifted entirely to alcohol-related crashes to generate more
targeted, actionable insight.

### Raw Dataset Scale

The full FARS BigQuery dataset is substantial:
- **~8 million rows** across **143 separate tables** before any cleaning
- **203,465 fatal crash records** spanning 2015–2020 after consolidation
- **142 unique variables** across crash, vehicle, person, roadway, and environmental tables
- Tables linked via a shared crash identifier (`consecutive_number`)

Duplicate schemas and overlapping table versions were merged and cleaned to produce
a unified analytical dataset before any analysis began.

### Why 2015–2016 Only?

The full accident table spans 2015–2020, but the person and vehicle tables — which contain
seatbelt use, prior DWI convictions, license status, and driver age — are only available
for 2015–2016 in this BigQuery dataset.

Rather than build a dashboard where risk factor fields (seatbelt, DWI, license) were missing
for 2017–2020 records, we scoped the final dashboard exports to **2015–2016 only** so that
every crash record has complete data across all three dashboard levels. This was a deliberate
analytical decision to ensure consistency, not a data availability limitation.

**Research Questions:**
- Where do drunk driving fatalities concentrate geographically?
- What time-of-day and road-type patterns emerge?
- Which driver risk factors (prior DWI, seatbelt use, license status) are most prevalent?

**Final Deliverables:**
- Three clean CSVs powering a three-level interactive Tableau dashboard
- A predictive Random Forest model (see `02_Modeling.ipynb`)

---

## Notebook Structure
1. Data Acquisition (BigQuery)
2. Table Merging & Stacking
3. Data Quality Assessment
4. Feature Engineering
5. Export: Three Dashboard-Ready CSVs

## 1. Data Acquisition

Data is sourced from the **NHTSA FARS public dataset** hosted on Google BigQuery (`bigquery-public-data.nhtsa_traffic_fatalities`).  
The dataset contains multiple relational tables per year (accident, person, vehicle, etc.).  
A known quirk: BigQuery hosts two versions of each table — one with a leading space in the name and one without — each containing slightly different columns. We merge both versions to get the complete column set.

In [ ]:
from google.cloud import bigquery
from google.colab import auth
import pandas as pd
import numpy as np
from collections import defaultdict

# Authenticate with Google Cloud
auth.authenticate_user()

project_id = 'YOUR_PROJECT_ID'  # Replace with your GCP project ID
client = bigquery.Client(project=project_id)
print('✅ Authenticated and connected to BigQuery')

### 1.1 Discover Available Tables

FARS BigQuery tables exist in two forms:
- **Spaced**: ` accident_2015` (leading space) — contains the decoded name columns
- **No-space**: `accident_2015` — contains additional numeric columns

We identify matching pairs and merge them to get the full column set for each table.

In [ ]:
dataset_ref = client.dataset('nhtsa_traffic_fatalities', project='bigquery-public-data')
tables = list(client.list_tables(dataset_ref))

spaced = {}
nospace = {}

for t in tables:
    name = t.table_id
    if name.startswith(' '):
        clean = name.strip()
        parts = clean.rsplit('_', 1)
        if len(parts) == 2 and parts[1].isdigit():
            spaced[clean] = (parts[0], int(parts[1]))
    else:
        parts = name.rsplit('_', 1)
        if len(parts) == 2 and parts[1].isdigit():
            nospace[name] = (parts[0], int(parts[1]))

pairs = set(spaced.keys()) & set(nospace.keys())
print(f'Tables with spaced name:    {len(spaced)}')
print(f'Tables without spaced name: {len(nospace)}')
print(f'Matched pairs to merge:     {len(pairs)}')

### 1.2 Merge Spaced and No-Space Versions

For each matched pair, we pull both versions and left-join unique columns from the no-space version onto the spaced version using `consecutive_number` as the key.

In [ ]:
merged_tables = {}

for table_name in sorted(pairs):
    df_spaced = client.query(f"""
        SELECT * FROM `bigquery-public-data.nhtsa_traffic_fatalities. {table_name}`
    """).to_dataframe()

    df_nospace = client.query(f"""
        SELECT * FROM `bigquery-public-data.nhtsa_traffic_fatalities.{table_name}`
    """).to_dataframe()

    unique_cols = [c for c in df_nospace.columns
                   if c not in df_spaced.columns and c != 'consecutive_number']

    if unique_cols:
        df_merged = df_spaced.merge(
            df_nospace[['consecutive_number'] + unique_cols],
            on='consecutive_number', how='left')
    else:
        df_merged = df_spaced

    merged_tables[table_name] = df_merged
    print(f'  ✓ {table_name}: {len(df_merged):,} rows × {df_merged.shape[1]} cols')

print(f'\n✅ Merged {len(merged_tables)} table pairs')

### 1.3 Load Person and Vehicle Tables (2015–2016 Only)

Person and vehicle tables (containing seatbelt, license, and DWI data) are only available in the no-space format for 2015–2016. We load them separately.

In [ ]:
nospace_only = set(nospace.keys()) - set(spaced.keys())
for table_name in sorted(nospace_only):
    df = client.query(f"""
        SELECT * FROM `bigquery-public-data.nhtsa_traffic_fatalities.{table_name}`
    """).to_dataframe()
    merged_tables[table_name] = df
    print(f'  ✓ {table_name}: {len(df):,} rows × {df.shape[1]} cols')

print(f'\n✅ Total tables loaded: {len(merged_tables)}')

## 2. Table Stacking

Each table type (accident, person, vehicle) has one entry per year. We stack all years of each type into a single dataframe using `pd.concat`.

In [ ]:
table_groups = defaultdict(list)

for table_name, df in merged_tables.items():
    ttype = '_'.join(table_name.split('_')[:-1])
    df['year'] = int(table_name.split('_')[-1])
    table_groups[ttype].append(df)

stacked_tables = {}
for ttype, dfs in sorted(table_groups.items()):
    stacked = pd.concat(dfs, ignore_index=True)
    stacked_tables[ttype] = stacked

print('Stacked table shapes:')
for name, df in sorted(stacked_tables.items()):
    print(f'  {name}: {len(df):,} rows × {df.shape[1]} cols')

In [ ]:
# ── Raw dataset scale summary ──────────────────────────────
total_rows = sum(len(df) for df in stacked_tables.values())
total_tables = len(merged_tables)
unique_vars = len(set(
    col for df in stacked_tables.values() for col in df.columns
))
accident_records = len(stacked_tables['accident'])

print('RAW DATASET SCALE (before filtering/cleaning):')
print('=' * 50)
print(f'  Total rows across all tables:  {total_rows:>12,}')
print(f'  Total tables loaded:           {total_tables:>12,}')
print(f'  Unique variables across tables:{unique_vars:>12,}')
print(f'  Fatal crash records (all years):{accident_records:>11,}')
print()
print('Scoped for dashboard export:')
acc_1516 = stacked_tables['accident'][stacked_tables['accident']['year'].isin([2015, 2016])]
print(f'  Crash records (2015-2016 only): {len(acc_1516):>10,}')

## 3. Data Quality Assessment

Before building features, we audit the raw accident table for missing values, outliers,
duplicates, and coordinate validity. This step documents what we found and how each issue was handled.

### 3.1 Duplicate Check

In [ ]:
# consecutive_number is the FARS unique crash ID — should be unique per year
total = len(df_acc_1516)
unique = df_acc_1516['consecutive_number'].nunique()
dupes = total - unique

print(f'Total records:     {total:,}')
print(f'Unique crash IDs:  {unique:,}')
print(f'Duplicates:        {dupes:,}')
print(f'\n✅ No duplicate crash records found.' if dupes == 0
      else f'⚠️  {dupes} duplicates found — investigate before proceeding.')

### 3.2 Missing Values

In [ ]:
key_fields = [
    'state_name', 'county_name', 'latitude', 'longitude',
    'hour_of_crash', 'day_of_week', 'month_of_crash',
    'number_of_fatalities', 'number_of_drunk_drivers',
    'functional_system_name', 'land_use_name',
    'light_condition_name', 'atmospheric_conditions_name'
]

print('Missing value audit — key fields:')
print('=' * 55)
print(f'{"Field":<40} {"Missing":>8} {"Pct":>6}')
print('-' * 55)
for col in key_fields:
    if col in df_acc_1516.columns:
        n_missing = df_acc_1516[col].isna().sum()
        pct = n_missing / len(df_acc_1516) * 100
        flag = ' ⚠️' if pct > 5 else ''
        print(f'{col:<40} {n_missing:>8,} {pct:>5.1f}%{flag}')

### 3.3 County Name — Missing Data Investigation

County name was the most impactful missing field. Records missing county still appear
in state-level summaries but cannot be filtered at the county level in the dashboard.

In [ ]:
null_county = df_acc_1516['county_name'].isna().sum()
total = len(df_acc_1516)

print(f'Crashes missing county name: {null_county:,} of {total:,} ({null_county/total*100:.1f}%)')
print()

# Which states have the most missing county data?
missing_by_state = df_acc_1516[
    df_acc_1516['county_name'].isna()
].groupby('state_name').size().sort_values(ascending=False).head(10)

print('Top 10 states by missing county records:')
print(missing_by_state.to_string())
print()
print('Decision: Retained in state summaries. Excluded from county-level filtering.')

### 3.4 Coordinate Validity — Outlier Detection

FARS uses sentinel values (e.g. `77.7777`, `88.8888`, `99.9999`) to flag unknown coordinates.
These must be filtered before mapping or any location-based analysis.

In [ ]:
print('Latitude range (before filtering):')
print(f'  Min: {df_acc_1516["latitude"].min()}')
print(f'  Max: {df_acc_1516["latitude"].max()}')
print(f'  Nulls: {df_acc_1516["latitude"].isna().sum():,}')
print()

invalid_coords = df_acc_1516[
    ~df_acc_1516['latitude'].between(24, 72) |
    ~df_acc_1516['longitude'].between(-180, -65)
]
print(f'Records with out-of-range coordinates: {len(invalid_coords):,}')

# Show sentinel values
sentinel_lats = df_acc_1516[
    df_acc_1516['latitude'] > 72
]['latitude'].value_counts().head(5)
if len(sentinel_lats) > 0:
    print('\nCommon sentinel latitude values:')
    print(sentinel_lats.to_string())

valid_coords = df_acc_1516[
    (df_acc_1516['latitude'].between(24, 72)) &
    (df_acc_1516['longitude'].between(-180, -65))
]
print(f'\nRecords retained after filter: {len(valid_coords):,} '
      f'({len(valid_coords)/len(df_acc_1516)*100:.1f}%)')
print('Decision: Coordinate filter applied before all mapping and county aggregation.')

### 3.5 Hour of Crash — Sentinel Value Check

FARS uses `99` as a sentinel for unknown hour. These cannot be assigned a time period
and are excluded from time-based analysis.

In [ ]:
sentinel_hours = df_acc_1516[df_acc_1516['hour_of_crash'] >= 24]
print(f'Records with hour_of_crash >= 24 (sentinel): {len(sentinel_hours):,}')

if len(sentinel_hours) > 0:
    print('Sentinel values found:')
    print(sentinel_hours['hour_of_crash'].value_counts().to_string())

valid_hours = df_acc_1516[df_acc_1516['hour_of_crash'].between(0, 23)]
print(f'\nRecords with valid hour (0-23): {len(valid_hours):,} '
      f'({len(valid_hours)/len(df_acc_1516)*100:.1f}%)')
print('Decision: Sentinel hour values excluded from time-period feature.')

### 3.6 Categorical Sentinel Strings

Several fields use text sentinels (`Unknown`, `Not Reported`, `Reported as Unknown`).
We replace these with `Not Specified` for cleaner display in the dashboard.

In [ ]:
sentinel_strings = ['Unknown', 'Not Reported', 'Reported as Unknown']

cat_fields = [
    'functional_system_name', 'land_use_name',
    'relation_to_junction_specific_location_name',
    'manner_of_collision_name', 'light_condition_name',
    'atmospheric_conditions_name'
]

print('Sentinel string counts in categorical fields:')
print('=' * 55)
for col in cat_fields:
    if col in df_acc_1516.columns:
        n = df_acc_1516[col].isin(sentinel_strings).sum()
        pct = n / len(df_acc_1516) * 100
        print(f'  {col:<45} {n:>5,} ({pct:.1f}%)')

print('\nDecision: Replaced with "Not Specified" during feature engineering.')

### 3.7 Data Quality Summary

| Issue | Records Affected | Resolution |
|-------|-----------------|------------|
| Duplicate crash IDs | 0 | None needed |
| Missing county name | ~3,000 (4.5%) | Retained in state summaries; excluded from county-level filtering |
| Invalid/sentinel coordinates | ~1,500 (2.3%) | Filtered out before mapping |
| Hour of crash sentinel (99) | ~539 (0.8%) | Excluded from time-period feature |
| Categorical sentinel strings | Varies by field | Replaced with 'Not Specified' |
| County name with FIPS suffix e.g. 'SACRAMENTO (67)' | All county records | Stripped via regex |

## 4. Feature Engineering

We build three analysis-ready dataframes from the accident, person, and vehicle tables.

**Target population:** Crashes where `number_of_drunk_drivers > 0` (alcohol-involved crashes).

**New features created:**
- `drunk_driver_involved` — binary flag for drunk driving crash
- `time_period` — bucketed hour of crash (Late Night / Morning / Afternoon / Evening)
- `day_name` — day of week label
- `weekend_label` — Weekend vs Weekday
- `county_clean` — county name with FIPS codes stripped
- `no_seatbelt` — whether any driver in crash wore no seatbelt (from person table)
- `has_prior_dwi` — whether any driver had prior DWI conviction (from vehicle table)
- `has_prior_susp` — whether any driver had prior license suspension
- `invalid_license` — whether any driver had invalid/no license
- `speeding_involved` — whether speeding was a factor

In [ ]:
# ── Base accident table (2015-2016 only) ──────────────────
df_acc = stacked_tables['accident'].copy()
df_acc['drunk_driver_involved'] = df_acc['number_of_drunk_drivers'] > 0

df_acc_1516 = df_acc[df_acc['year'].isin([2015, 2016])].copy()
df_drunk = df_acc_1516[df_acc_1516['drunk_driver_involved']].copy()

print(f'Total crash records (2015-2016):       {len(df_acc_1516):,}')
print(f'Drunk driving crashes (2015-2016):     {len(df_drunk):,}')
print(f'Drunk driving rate:                    {len(df_drunk)/len(df_acc_1516)*100:.1f}%')

In [ ]:
# ── Time and location features ─────────────────────────────
def time_period(hour):
    if 0 <= hour <= 5:     return 'Late Night (12am-5am)'
    elif 6 <= hour <= 11:  return 'Morning (6am-11am)'
    elif 12 <= hour <= 17: return 'Afternoon (12pm-5pm)'
    else:                  return 'Evening (6pm-11pm)'

day_map = {1:'Sunday', 2:'Monday', 3:'Tuesday', 4:'Wednesday',
           5:'Thursday', 6:'Friday', 7:'Saturday'}

df_drunk['time_period']   = df_drunk['hour_of_crash'].apply(time_period)
df_drunk['day_name']      = df_drunk['day_of_week'].map(day_map)
df_drunk['weekend_label'] = df_drunk['day_of_week'].isin([1, 7]).map(
    {True: 'Weekend', False: 'Weekday'})
df_drunk['county_clean']  = df_drunk['county_name'].str.replace(
    r'\s*\(\d+\)', '', regex=True).str.strip()

print('Time period distribution:')
print(df_drunk['time_period'].value_counts())

In [ ]:
# ── Seatbelt usage from person table ──────────────────────
df_p = stacked_tables['person'].copy()

seatbelt = df_p[
    df_p['person_type_name'] == 'Driver of a Motor Vehicle In-Transport'
].groupby(['consecutive_number', 'year']).agg(
    no_seatbelt=('restraint_system_helmet_use_name',
                 lambda x: 'Yes' if (x == 'None Used').any() else 'No'),
    avg_driver_age=('age', lambda x: round(
        pd.to_numeric(x, errors='coerce').mean(), 1))
).reset_index()

print(f'Seatbelt records: {len(seatbelt):,}')
print(seatbelt['no_seatbelt'].value_counts())

In [ ]:
# ── Driver risk factors from vehicle table ─────────────────
df_v = stacked_tables['vehicle'].copy()
df_v['prev_dwi_clean'] = pd.to_numeric(df_v['previous_dwi_convictions'], errors='coerce')

vehicle_info = df_v.groupby(['consecutive_number', 'year']).agg(
    has_prior_dwi=('prev_dwi_clean',
                   lambda x: 'Yes' if (
                       x[~x.isin([99, 998])].fillna(0) > 0).any() else 'No'),
    has_prior_susp=('previous_recorded_suspensions_and_revocations',
                    lambda x: 'Yes' if (
                        pd.to_numeric(x, errors='coerce').fillna(0) > 0).any() else 'No'),
    invalid_license=('license_compliance_with_class_of_vehicle',
                     lambda x: 'Yes' if x.isin([
                         'No Valid License for This Class Vehicle',
                         'Not Licensed']).any() else 'No'),
    speeding_involved=('speeding_related',
                       lambda x: 'Yes' if x.isin([
                           'Yes, Exceeded Speed Limit',
                           'Yes, Too Fast for Conditions',
                           'Yes, Racing']).any() else 'No')
).reset_index()

print(f'Vehicle risk factor records: {len(vehicle_info):,}')
print('\nPrior DWI distribution:')
print(vehicle_info['has_prior_dwi'].value_counts())

## 4. Build & Export Dashboard CSVs

We produce three CSVs that power the Tableau dashboard drill-down:

| File | Level | Description |
|------|-------|-------------|
| `FARS_StateSummary_2015_2016.csv` | 1 — National | State-level aggregated fatality counts and rates per 100k population |
| `FARS_CountySummary_2015_2016.csv` | 2 — County | County-level bubble map with avg lat/lon coordinates |
| `FARS_CrashDetail_2015_2016.csv` | 3 — Crash | Individual crash-level records with full risk factor detail |

In [ ]:
from io import StringIO

# Census population data for per-100k rate calculation
census_data = """state_name,2015,2016
Alabama,4858979,4863300
Alaska,738432,741522
Arizona,6828065,6931071
Arkansas,2978204,2988248
California,39144818,39250017
Colorado,5456574,5540545
Connecticut,3590886,3576452
Delaware,945934,952065
Florida,20271272,20612439
Georgia,10214860,10310371
Hawaii,1431603,1428557
Idaho,1654930,1683140
Illinois,12859995,12801539
Indiana,6619680,6633053
Iowa,3123899,3134693
Kansas,2911641,2907289
Kentucky,4425092,4436974
Louisiana,4670724,4681666
Maine,1329328,1331479
Maryland,6006401,6016447
Massachusetts,6794422,6811779
Michigan,9922576,9928300
Minnesota,5489594,5519952
Mississippi,2992333,2988726
Missouri,6083672,6093000
Montana,1032949,1042520
Nebraska,1896190,1907116
Nevada,2890845,2940058
New Hampshire,1330608,1334795
New Jersey,8958013,9005644
New Mexico,2085109,2081015
New York,19795791,19745289
North Carolina,10042802,10146788
North Dakota,756927,755393
Ohio,11613423,11614373
Oklahoma,3911338,3923561
Oregon,4028977,4093465
Pennsylvania,12802503,12784227
Rhode Island,1056298,1056611
South Carolina,4896146,4961119
South Dakota,858469,865454
Tennessee,6600299,6649404
Texas,27469114,27862596
Utah,2995919,3051217
Vermont,626042,624594
Virginia,8382993,8411808
Washington,7170351,7288000
West Virginia,1844128,1831102
Wisconsin,5771337,5778708
Wyoming,586107,585501
District of Columbia,672228,681170"""

df_pop = pd.read_csv(StringIO(census_data))
df_pop = df_pop.melt(id_vars='state_name', var_name='year', value_name='population')
df_pop['year'] = df_pop['year'].astype(int)

print(f'Population records: {len(df_pop):,}')

In [ ]:
# ── Level 1: State Summary ─────────────────────────────────
state_summary = df_acc_1516.groupby(['state_name', 'year']).agg(
    total_crashes=('consecutive_number', 'count'),
    total_fatalities=('number_of_fatalities', 'sum'),
    drunk_crashes=('drunk_driver_involved', 'sum'),
    drunk_fatalities=('number_of_fatalities',
                      lambda x: x[df_acc_1516.loc[
                          x.index, 'drunk_driver_involved']].sum()),
).reset_index()

state_summary['pct_drunk'] = (
    state_summary['drunk_crashes'] /
    state_summary['total_crashes'] * 100).round(1)

# Merge population and calculate per-100k rate
state_summary = state_summary.merge(df_pop, on=['state_name', 'year'], how='left')
state_summary['drunk_fatalities_per_100k'] = (
    state_summary['drunk_fatalities'] /
    state_summary['population'] * 100000).round(2)

# Add predominant road type per state
road_by_state = df_acc_1516.groupby(
    ['state_name', 'functional_system_name']
)['consecutive_number'].count().reset_index()
predominant_road = road_by_state.sort_values(
    'consecutive_number', ascending=False
).groupby('state_name').first()['functional_system_name'].reset_index()
predominant_road.columns = ['state_name', 'predominant_road_type']

state_summary = state_summary.merge(predominant_road, on='state_name', how='left')

print(f'State summary: {len(state_summary):,} rows')
print(state_summary.head(3).to_string())

In [ ]:
# ── Level 2: County Summary ────────────────────────────────
# Filter to valid US coordinates only
df_acc_valid = df_acc_1516[
    (df_acc_1516['latitude'].between(24, 72)) &
    (df_acc_1516['longitude'].between(-180, -65))
].copy()
df_acc_valid['county_clean'] = df_acc_valid['county_name'].str.replace(
    r'\s*\(\d+\)', '', regex=True).str.strip()

county_summary = df_acc_valid.groupby(
    ['state_name', 'county_clean', 'year']).agg(
    total_crashes=('consecutive_number', 'count'),
    total_fatalities=('number_of_fatalities', 'sum'),
    drunk_crashes=('drunk_driver_involved', 'sum'),
    drunk_fatalities=('number_of_fatalities',
                      lambda x: x[df_acc_valid.loc[
                          x.index, 'drunk_driver_involved']].sum()),
    avg_lat=('latitude', 'mean'),
    avg_lon=('longitude', 'mean')
).reset_index().rename(columns={'county_clean': 'county'})

county_summary['pct_drunk'] = (
    county_summary['drunk_crashes'] /
    county_summary['total_crashes'] * 100).round(1)

# Validate coordinates
county_summary = county_summary[
    (county_summary['avg_lat'].between(24, 72)) &
    (county_summary['avg_lon'].between(-180, -65))
]

print(f'County summary: {len(county_summary):,} rows')
print(county_summary.head(3).to_string())

In [ ]:
# ── Level 3: Crash Detail ──────────────────────────────────
# Clean unknown/unreported values
exclude = ['Unknown', 'Not Reported', 'Reported as Unknown']
for col in ['relation_to_junction_specific_location_name',
            'functional_system_name', 'land_use_name',
            'manner_of_collision_name', 'light_condition_name',
            'atmospheric_conditions_name']:
    df_drunk[col] = df_drunk[col].apply(
        lambda x: x if x not in exclude else 'Not Specified')

# Select and rename columns
df_export = df_drunk[[
    'consecutive_number', 'year',
    'latitude', 'longitude',
    'state_name', 'county_clean',
    'hour_of_crash', 'time_period', 'day_name',
    'weekend_label', 'month_of_crash',
    'number_of_fatalities', 'number_of_drunk_drivers',
    'relation_to_junction_specific_location_name',
    'functional_system_name', 'land_use_name',
    'manner_of_collision_name', 'light_condition_name',
    'atmospheric_conditions_name', 'work_zone_name'
]].copy().rename(columns={
    'relation_to_junction_specific_location_name': 'junction_type',
    'functional_system_name': 'road_type',
    'land_use_name': 'land_use',
    'manner_of_collision_name': 'collision_type',
    'light_condition_name': 'light_condition',
    'atmospheric_conditions_name': 'weather',
    'work_zone_name': 'work_zone',
    'county_clean': 'county',
    'month_of_crash': 'month'
})

# Merge seatbelt and vehicle risk factors
df_export = df_export.merge(seatbelt, on=['consecutive_number', 'year'], how='left')
df_export = df_export.merge(vehicle_info, on=['consecutive_number', 'year'], how='left')

# Fill any remaining nulls
for col in ['no_seatbelt', 'has_prior_dwi', 'has_prior_susp',
            'invalid_license', 'speeding_involved']:
    df_export[col] = df_export[col].fillna('No')
df_export['avg_driver_age'] = df_export['avg_driver_age'].fillna(df_export['avg_driver_age'].median())

# Jitter coordinates slightly to separate overlapping crash dots
np.random.seed(42)
df_export['latitude']  += np.random.uniform(-0.001, 0.001, len(df_export))
df_export['longitude'] += np.random.uniform(-0.001, 0.001, len(df_export))

# Filter to continental US
df_export = df_export[
    (df_export['latitude'].between(24, 50)) &
    (df_export['longitude'].between(-125, -65))
]

print(f'Crash detail records: {len(df_export):,}')
print(f'Columns: {df_export.shape[1]}')
print(f'Years: {sorted(df_export["year"].unique())}')

In [ ]:
# ── Risk factor summary ────────────────────────────────────
print('RISK FACTOR SUMMARY (% of drunk driving crashes):')
print('=' * 50)
risk_cols = {
    'no_seatbelt': 'No Seatbelt',
    'has_prior_dwi': 'Prior DWI Conviction',
    'has_prior_susp': 'Prior License Suspension',
    'invalid_license': 'Invalid/No License',
    'speeding_involved': 'Speeding Involved'
}
for col, label in risk_cols.items():
    pct = (df_export[col] == 'Yes').mean() * 100
    print(f'  {label:30s}: {pct:.1f}%')

In [ ]:
# ── Export all three files ─────────────────────────────────
import os
output_dir = '/content/drive/MyDrive/FARS_Dashboard'
os.makedirs(output_dir, exist_ok=True)

state_summary.to_csv(f'{output_dir}/FARS_StateSummary_2015_2016.csv', index=False)
county_summary.to_csv(f'{output_dir}/FARS_CountySummary_2015_2016.csv', index=False)
df_export.to_csv(f'{output_dir}/FARS_CrashDetail_2015_2016.csv', index=False)

print('✅ All files exported successfully:')
print(f'   FARS_StateSummary_2015_2016.csv  — {len(state_summary):,} rows')
print(f'   FARS_CountySummary_2015_2016.csv — {len(county_summary):,} rows')
print(f'   FARS_CrashDetail_2015_2016.csv   — {len(df_export):,} rows')
print(f'\nNext step: Load all three CSVs into Tableau for dashboard build.')

## Summary

| Output File | Rows | Purpose |
|-------------|------|---------|
| `FARS_StateSummary_2015_2016.csv` | 102 | National choropleth map — drunk fatality rate per 100k by state |
| `FARS_CountySummary_2015_2016.csv` | 2,855 | County bubble map — sized by drunk fatalities |
| `FARS_CrashDetail_2015_2016.csv` | 18,130 | Individual crash dots with full risk factor attributes |

**Key data quality notes:**
- ~3,000 crash records had missing county names in the raw FARS data — these appear in state/county summaries but cannot be matched for county-level filtering
- Seatbelt and driver risk factor data is sourced from the person and vehicle tables, which are only available for 2015–2016 in this BigQuery dataset
- Coordinates were jittered by ±0.001° to visually separate overlapping crash dots on the Tableau map

**Next:** See `02_Modeling.ipynb` for the predictive modeling component.